# Deepfake Group 31 - Preprocessing
This notebook mounts Google Drive, prepares a reproducible dataset pipeline, detects faces with MTCNN, and visualizes preprocessed samples.

In [ ]:
from google.colab import drive
import os, random, numpy as np, torch

drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/Deepfake_Group31'
SEED = 42
os.makedirs(BASE_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(f"BASE_DIR set to: {BASE_DIR}")

## Install required dependencies
This cell installs all libraries requested for preprocessing and model development.

In [ ]:
!pip install -q torch torchvision facenet-pytorch opencv-python scikit-learn matplotlib tqdm wandb

## Import libraries
Load all required modules for dataset building, transforms, face detection, and plotting.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from torchvision import transforms
from facenet_pytorch import MTCNN
from tqdm.auto import tqdm

## Define dataset class
This dataset reads images from `real/` and `fake/` folders and maps labels as `real=0`, `fake=1`.

In [ ]:
class DeepfakeImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.samples = []
        class_map = {'real': 0, 'fake': 1}

        for class_name, label in class_map.items():
            class_dir = os.path.join(root_dir, class_name)
            if not os.path.isdir(class_dir):
                continue
            for file_name in sorted(os.listdir(class_dir)):
                if file_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.webp')):
                    self.samples.append((os.path.join(class_dir, file_name), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_path, label = self.samples[idx]
        image = Image.open(image_path).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        return image, label, image_path

## Configure transforms and load dataset
Images are resized to `128x128` and normalized using mean/std `0.5` for each channel.

In [ ]:
data_dir = os.path.join(BASE_DIR, 'data')
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

full_dataset = DeepfakeImageDataset(data_dir, transform=transform)
print(f"Total discovered images: {len(full_dataset)}")

## Create reproducible 80/10/10 train-val-test splits
This uses `random_split` with seed `42`, targeting 10,000 images when available.

In [ ]:
target_total = 10000
available = len(full_dataset)
if available == 0:
    raise ValueError(f"No images found under {data_dir}/real and {data_dir}/fake")

if available >= target_total:
    generator = torch.Generator().manual_seed(SEED)
    selected_indices = torch.randperm(available, generator=generator)[:target_total].tolist()
    working_dataset = Subset(full_dataset, selected_indices)
    split_lengths = [8000, 1000, 1000]
else:
    working_dataset = full_dataset
    train_len = int(0.8 * available)
    val_len = int(0.1 * available)
    test_len = available - train_len - val_len
    split_lengths = [train_len, val_len, test_len]

generator = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset, test_dataset = random_split(working_dataset, split_lengths, generator=generator)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train/Val/Test sizes: {len(train_dataset)}/{len(val_dataset)}/{len(test_dataset)}")

## Detect and crop faces using MTCNN
This helper applies face detection per image and returns cropped face tensors.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
mtcnn = MTCNN(image_size=128, margin=10, keep_all=False, device=device)

def detect_and_crop_faces(batch_images):
    faces = []
    for tensor_img in batch_images:
        np_img = ((tensor_img.permute(1, 2, 0).cpu().numpy() * 0.5) + 0.5).clip(0, 1)
        pil_img = Image.fromarray((np_img * 255).astype('uint8'))
        face = mtcnn(pil_img)
        if face is None:
            face = transform(pil_img)
        else:
            face = transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])(face)
        faces.append(face)
    return torch.stack(faces)

## Visualize 16 preprocessed face samples
This cell shows a 4x4 grid of cropped/normalized faces with class labels.

In [ ]:
batch_images, batch_labels, _ = next(iter(train_loader))
face_batch = detect_and_crop_faces(batch_images[:16])

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for i, ax in enumerate(axes.flatten()):
    img = face_batch[i].permute(1, 2, 0).cpu().numpy()
    img = (img * 0.5) + 0.5
    label_name = 'FAKE' if int(batch_labels[i]) == 1 else 'REAL'
    ax.imshow(img.clip(0, 1))
    ax.set_title(label_name)
    ax.axis('off')
plt.tight_layout()
plt.show()